In [4]:
from nanogpt.layers import MultiHeadAttention, Attention, Embedding, FeedForward, LayerNorm, AliBiPositionalEncoding, RMSNorm
from nanogpt import Transformer, TransformerBlock
from nanogpt import Indicer, BPETokenizer
from nanogpt import SlidingWindow
from nanogpt.loss_functions import CrossEntropyLoss, BradleyTerryLoss
from nanogpt import Optimizer
import numpy as np
import cupy as cp
import gc
import json
import re
import random
import math
import time
import os
import copy
import requests
from reversegradGPU import Tensor
from collections import Counter
from nanogpt_utils import (
    preprocess_text, split_on,
    collect, sync, free_pool, pool_usage,
    save_model, load_model,
    get_batches, pad_sequences, generate_mask,
    call_gemma, parse_pairs, parse_pairs_reward,
    proportion_ci, n_to_beat_chance, paired_diff,
)

## Data Loading & Preprocessing

In [6]:
path_to_text = "D:/NanoGPT_adam/data/TinyStories-train.txt"
with open(path_to_text, 'r', encoding='utf-8') as f:
    text = f.read(400000000)
processed_text = preprocess_text(text)
print("Processed text:", processed_text[:100])


Processed text: one day ,  a little girl named lily found a needle in her room .  she knew it was difficult to play 


In [7]:
del text
gc.collect()


257

## Tokenization & Encoding

In [8]:
token_map = processed_text.split()
boundry_tokens = ["<|storystart|>", "<|storyend|>", "<|instructionstart|>", "<|instructionend|>",
                  "<|modelstart|>", "<|modelend|>", "<|pad|>", "retell"]
indicer = Indicer(vocab_size=10000, gaps=boundry_tokens)
map_indicer = indicer.fit(list(token_map))
np.save("D:/NanoGPT_adam/data/indices.npy", map_indicer)
del map_indicer; gc.collect()
vocab_size = 10008
print("Vocabulary size:", vocab_size)


Vocabulary size: 10008


In [6]:
storystart     = indicer.encode(["<|storystart|>"])[0]
storyend       = indicer.encode(["<|storyend|>"])[0]
instructionstart = indicer.encode(["<|instructionstart|>"])[0]
instructionend = indicer.encode(["<|instructionend|>"])[0]
modelstart     = indicer.encode(["<|modelstart|>"])[0]
modelend       = indicer.encode(["<|modelend|>"])[0]
pad            = indicer.encode(["<|pad|>"])[0]


In [69]:
split_index = int(len(token_map) * 0.8)
train_token_map = token_map[:split_index]
test_token_map  = token_map[split_index:]
while train_token_map[-1] != "<|endoftext|>": train_token_map.pop()
while test_token_map[0]  != "<|endoftext|>": test_token_map.pop(0)
test_token_map.pop(0)
encoded_train_tokens = np.fromiter(
    (indicer.map.get(t, 0) for t in train_token_map),
    dtype="int16", count=len(train_token_map)
)
encoded_test_tokens = np.fromiter(
    (indicer.map.get(t, 0) for t in test_token_map),
    dtype="int16", count=len(test_token_map)
)
print("Train tokens:", len(train_token_map))
print("Test tokens:",  len(test_token_map))


Train tokens: 75201574
Test tokens: 18800195


In [7]:
del processed_text, token_map
gc.collect()


0

## Utilities

In [12]:
def train_loop_gpu(model, data, loss_log, iters=20, lr=0.00015, gc_frequency=10, batch=2):
    loss_func = CrossEntropyLoss()
    optimizer = Optimizer(model)
    for i in range(iters):
        sync(); t = time.perf_counter()
        optimizer.zero_grad()
        x, y = get_batches(data, batch)
        output = model.forward(x)
        loss = loss_func.forward(output, y)
        loss_val = float(loss.data)
        loss.backward()
        optimizer.step(lr)
        collect(loss)
        del output, loss, x, y
        if i % gc_frequency == 0:
            gc.collect()
        loss_log.append(loss_val)
        sync(); t_total = time.perf_counter() - t
        print(f" Iteration {i}: {t_total:.3f}s  loss: {loss_val:.4f}")
    return np.asarray(loss_log)


In [13]:
def generate(model, tokens, indicer, max_tokens=20, temperature=0.5):
    text = list(indicer.encode(list(tokens)))
    for _ in range(max_tokens):
        output = model.forward(text).data
        logits = cp.asnumpy(output[-1]) / temperature
        logits -= logits.max()
        p = np.exp(logits); p /= p.sum()
        text.append(np.random.choice(len(p), p=p))
    return text

def generate_eval(model, prompt_seq, max_tokens=30, temperature=0.5):
    text = list(prompt_seq)
    for _ in range(max_tokens):
        out = model.forward(text)
        logits = cp.asnumpy(out.data[-1])
        collect(out); del out
        logits = logits / temperature
        logits -= logits.max()
        p = np.exp(logits); p /= p.sum()
        tok = int(np.random.choice(len(p), p=p))
        if tok == modelend:
            break
        text.append(tok)
    return text[len(prompt_seq):]


## SFT Training Loops

In [ ]:
def get_pairs(n_groups, stories, verbose=False):
    batch = []
    stories_chosen = random.sample(stories, k=n_groups)
    for i, story in enumerate(stories_chosen):
        stories.remove(story)
        if verbose and i % verbose == 0:
            print(f"Pair {i} generated")
        for pair in parse_pairs(sample(" ".join(story))):
            seq = [storystart] + list(indicer.encode(story)) + [storyend, instructionstart]
            seq += indicer.encode(preprocess_text(pair[1]).split())
            seq += [instructionend, modelstart]
            seq += indicer.encode(preprocess_text(pair[2]).split())
            seq.append(modelend)
            batch.append(seq)
    return batch, stories


In [ ]:
# Wave-based SFT (online generation + training)
def train_loop_sft_wave(model, stories, acheive_loss, loss_log=[], lr=0.00015,
                        gc_frequency=10, batch_size=2, wave_size=100,
                        max_waves=None, checkpoint=None, checkpoint_name="model", verbose=False):
    loss_func = CrossEntropyLoss()
    optimizer = Optimizer(model)
    waves_done = 0
    while True:
        wave_loss = []
        wave, stories = get_pairs(wave_size, stories, verbose=verbose)
        batches = [pad_sequences(wave[i:i+batch_size], pad) for i in range(0, len(wave), batch_size)]
        for i, batch in enumerate(batches):
            optimizer.zero_grad()
            sync(); t = time.perf_counter()
            x, y, mask = [], [], []
            for seq in batch:
                x.append(seq[:-1]); y.append(seq[1:])
                mask.append(generate_mask(seq, instructionstart, instructionend, modelend)[1:])
            x, y, mask = cp.array(x), cp.array(y), cp.array(mask)
            prediction = model.forward(x)
            loss = loss_func.forward(prediction, y, mask)
            loss_val = float(loss.data)
            loss.backward()
            optimizer.step(lr)
            collect(loss)
            del prediction, loss, x, y, mask
            if i % gc_frequency == 0:
                gc.collect()
            loss_log.append(loss_val); wave_loss.append(loss_val)
            if verbose and i % verbose == 0:
                print(f" Iter {i}: {time.perf_counter()-t:.3f}s  loss: {loss_val:.4f}")
        del batches
        gc.collect(); cp.get_default_memory_pool().free_all_blocks()
        waves_done += 1
        if waves_done == max_waves or cp.mean(cp.array(wave_loss[-20:])) < acheive_loss: break
        if checkpoint is not None and waves_done % checkpoint == 0:
            save_model(model, f"D:/Nanogpt-SFT/models/{checkpoint_name}_wave{waves_done}.npy")
            print("Checkpoint saved")
    return loss_log, stories


In [ ]:
# File-based SFT (pre-generated pairs from disk)
def train_loop_sft(model, pairs, loss_log=[], lr=0.00015, gc_frequency=10, batch_size=2):
    loss_func = CrossEntropyLoss()
    optimizer = Optimizer(model)
    batches = [pad_sequences(pairs[i:i+batch_size], pad) for i in range(0, len(pairs), batch_size)]
    for i, batch in enumerate(batches):
        optimizer.zero_grad()
        sync(); t = time.perf_counter()
        x, y, mask = [], [], []
        for seq in batch:
            x.append(seq[:-1]); y.append(seq[1:])
            mask.append(generate_mask(seq, instructionstart, instructionend, modelend)[1:])
        x, y, mask = cp.array(x), cp.array(y), cp.array(mask)
        prediction = model.forward(x)
        loss = loss_func.forward(prediction, y, mask)
        loss_val = float(loss.data)
        loss.backward()
        optimizer.step(lr)
        collect(loss)
        del prediction, loss, x, y, mask
        if i % gc_frequency == 0:
            gc.collect(); cp.get_default_memory_pool().free_all_blocks()
        loss_log.append(loss_val)
        sync(); t_total = time.perf_counter() - t
        print(f" Iter {i}: {t_total:.3f}s  loss: {loss_val:.4f}")
    del batches
    gc.collect(); cp.get_default_memory_pool().free_all_blocks()
    return loss_log


## Base Model

In [ ]:
path = "D:/Nanogpt-SFT/models/base_model_500.npy"
model = Transformer(vocab_size, 768, 6, 4, 8, rope=True, pre_ln=False, rms=True, swiglu=False)
try:
    model = load_model(model, path)
    print("Base model loaded.")
except FileNotFoundError:
    while not os.path.exists(path):
        proceed = input(f"Model not found at {path}. Enter new path (y) or train (n): ")
        if proceed == "y": path = input("New path: ")
        else: break
    if os.path.exists(path):
        model = load_model(model, path)
    else:
        model_loss = train_loop_gpu(model, encoded_train_tokens, [], iters=500, gc_frequency=10, batch=3)
        save_model(model, path)


In [71]:
stories = split_on(train_token_map, "<|endoftext|>")
del train_token_map, encoded_train_tokens, encoded_test_tokens
gc.collect()
print(len(stories), "train stories")


350466 train stories


In [49]:
del train_token_map, encoded_train_tokens, encoded_test_tokens
gc.collect()

0

In [ ]:
def sample_reward(story):
    example = """Question[request]: What was Lily able to sew a button on with the needle?
    Good Response: She wanted to sew a button onto her shirt, making it look nice again.
    Bad Response: Lily had a box and it was full of toys, she put the needle into the box.

    Question[retell]: Retell the part where Lily asked her mom for help with the needle.
    Good Response: Lily went to her mom and asked if they could share the needle to fix her shirt.
    Bad Response: Jack asked his dad if he can play with the big red ball, his dad said no.

    Question[complete]: Lily and her mom sewed a ______
    Good Response: button onto her shirt, and it was not difficult because they helped each other.
    Bad Response: patch onto her shirt, it was difficult because they had to be careful with the needle"""


    prompt = f"\nHere is a children's story:\n{story}\nWrite 3 instruction, good response and bad response pairs based on this story. The response must be written in the same simple third-person storybook style as the story above. Use ONLY simple words that appear in stories like this one - no word a five-year-old would not know. Format as:\nQuestion[type]: ...\nGood Response: ...\nBad Response: ...\nEach pair must be one of these types: ask a question about what happened in the story(type=request), retell part of the story in a different way(type=retell) and finish an incomplete sentence from it(type=complete), you must not ask any deductions or inference. The response must be at least 10 words long. For the question pair, the question must be directly in the story, no deductions or inference. You also can not invent new stories(e.g. what happened next)\nEXAMPLE OUTPUT FORMAT match the format exactly, and output nothing else: {example}  Every type must be used once(3 types and 3 pairs), never 2 pairs that has the same type, request pairs must start with what/who/where/when/how. For the bad response, you can either answer the question wrongly(make it obvious) or answer a completely different question(inventing things). \n\n I have a base model training on tinystories and I want to do SFT on it. The models goal is to use the information in front of it to respond to a question, NOT invent new content. Assume that the model that is going to use your pair has already seen the story that I gave you."
    text = call_gemma(prompt)
    return text

In [72]:
# Save once
import json
with open("D:/Nanogpt-SFT/data/stories.jsonl", "w") as f:
    for story in stories:
        f.write(json.dumps(story) + "\n")
del stories
gc.collect()

0

## Slice 1 — Reward Model

In [30]:
def get_preference_pairs(n, stories_path, verbose=False, write_path=None, return_pairs=True):
    if not return_pairs and write_path is None:
        raise ValueError("return_pairs=False requires a write_path")
    with open(stories_path) as f:
        total = sum(1 for _ in f)
    chosen_indices = set(random.sample(range(total), n))
    good_batch, bad_batch = [], []
    i = 0
    with open(stories_path) as f:
        for line_idx, line in enumerate(f):
            if line_idx not in chosen_indices:
                continue
            story = json.loads(line)
            if verbose and i % verbose == 0:
                print(f"Group {i} finished generating")
            i += 1
            for pair in parse_pairs_reward(sample_reward(" ".join(story))):
                if len(pair) < 4:
                    continue
                seq = [storystart] + list(indicer.encode(story)) + [storyend, instructionstart]
                seq += indicer.encode(preprocess_text(pair[1]).split())
                seq += [instructionend, modelstart]
                bad_seq = seq.copy()
                seq += indicer.encode(preprocess_text(pair[2]).split())
                bad_seq += indicer.encode(preprocess_text(pair[3]).split())
                seq.append(modelend)
                bad_seq.append(modelend)
                if len(seq) <= 400 and len(bad_seq) <= 400:
                    if write_path is not None:
                        with open(write_path, "a") as f:
                            f.write(json.dumps({"good": seq, "bad": bad_seq}) + "\n")
                    if return_pairs:
                        good_batch.append(seq)
                        bad_batch.append(bad_seq)
    if return_pairs:
        return good_batch, bad_batch

In [27]:
import sys
big = {k: sys.getsizeof(v) for k, v in globals().items() if sys.getsizeof(v) > 10_000_000}
print(sorted(big.items(), key=lambda x: -x[1]))

[('test_token_map', 150403248)]


In [ ]:
# Wave-based RM training (online generation + training)
def reward_model_train_wave(model, gen_model, stories, max_waves, loss_log=[],
                            lr=0.00015, gc_frequency=10, batch_size=3,
                            wave_size=100, acheive_loss=None,
                            checkpoint=None, checkpoint_name="reward_model", verbose=False):
    optimizer = Optimizer(model)
    loss_func = BradleyTerryLoss()
    waves_done = 0
    while True:
        wave_loss = []
        good_wave, bad_wave = get_preference_pairs(wave_size, stories, gen_model, verbose=verbose)
        good_batches = [good_wave[i:i+batch_size] for i in range(0, len(good_wave), batch_size)]
        bad_batches  = [bad_wave[i:i+batch_size]  for i in range(0, len(bad_wave),  batch_size)]
        for i, good_batch, bad_batch in zip(range(len(good_batches)), good_batches, bad_batches):
            optimizer.zero_grad()
            sync(); t = time.perf_counter()
            good_lens = [len(s)-1 for s in good_batch]
            bad_lens  = [len(s)-1 for s in bad_batch]
            good_batch = cp.asarray(pad_sequences(good_batch, pad))
            bad_batch  = cp.asarray(pad_sequences(bad_batch, pad))
            out_good = model.forward(good_batch, good_lens)
            out_bad  = model.forward(bad_batch,  bad_lens)
            loss = loss_func.forward(out_good, out_bad)
            loss_val = float(loss.data)
            loss.backward()
            optimizer.step(lr)
            collect(loss)
            del out_good, out_bad, loss, good_batch, bad_batch
            cp.get_default_memory_pool().free_all_blocks()
            if i % gc_frequency == 0: gc.collect()
            loss_log.append(loss_val); wave_loss.append(loss_val)
            if verbose and i % verbose == 0:
                print(f"Iter {i}  loss: {loss_val:.4f}  time: {time.perf_counter()-t:.3f}s")
        del good_batches, bad_batches, good_wave, bad_wave
        gc.collect(); cp.get_default_memory_pool().free_all_blocks()
        waves_done += 1
        if waves_done == max_waves or (acheive_loss is not None and cp.mean(cp.array(wave_loss[-20:])) < acheive_loss): break
        if checkpoint is not None and waves_done % checkpoint == 0:
            save_model(model, f"D:/Nanogpt-SFT/models/{checkpoint_name}_wave{waves_done}.npy")
            print("Checkpoint saved")
    return loss_log


In [10]:
# File-based RM training (pre-generated pairs from disk)
def reward_model_train(model, good_pairs, bad_pairs, loss_log=[],
                       lr=0.00015, gc_frequency=10, batch_size=3):
    optimizer = Optimizer(model)
    loss_func = BradleyTerryLoss()
    good_batches = [good_pairs[i:i+batch_size] for i in range(0, len(good_pairs), batch_size)]
    bad_batches  = [bad_pairs[i:i+batch_size]  for i in range(0, len(bad_pairs),  batch_size)]
    for i, good_batch, bad_batch in zip(range(len(good_batches)), good_batches, bad_batches):
        optimizer.zero_grad()
        sync(); t = time.perf_counter()
        good_lens = [len(s)-1 for s in good_batch]
        bad_lens  = [len(s)-1 for s in bad_batch]
        good_batch = cp.asarray(pad_sequences(good_batch, pad))
        bad_batch  = cp.asarray(pad_sequences(bad_batch, pad))
        out_good = model.forward(good_batch, good_lens)
        out_bad  = model.forward(bad_batch,  bad_lens)
        loss = loss_func.forward(out_good, out_bad)
        loss_val = float(loss.data)
        loss.backward()
        optimizer.step(lr)
        collect(loss)
        collect(out_good)
        del out_good, out_bad, loss, good_batch, bad_batch
        cp.get_default_memory_pool().free_all_blocks()
        if i % gc_frequency == 0: gc.collect()
        loss_log.append(loss_val)
        print(f"Iter {i}  loss: {loss_val:.4f}  time: {time.perf_counter()-t:.3f}s")
    return loss_log


In [36]:
gc.collect()
cp.get_default_memory_pool().free_all_blocks()

## Generate Eval Pairs (run once, from test_token_map)

In [73]:
# Run this cell once to generate the held-out eval set.
# Requires test_token_map still alive (don't del it before running this).
test_stories = split_on(test_token_map, "<|endoftext|>")
# Save once
import json
with open("D:/Nanogpt-SFT/data/test_stories.jsonl", "w") as f:
    for story in test_stories:
        f.write(json.dumps(story) + "\n")
del test_stories
gc.collect()

0

In [74]:

get_preference_pairs(9, "D:/Nanogpt-SFT/data/stories.jsonl",
                     write_path="D:/Nanogpt-SFT/data/eval_preference_pairs.jsonl",
                     return_pairs=False, verbose=2)
print("Eval pairs written.")


Group 0 finished generating
Group 2 finished generating
Group 4 finished generating
Group 6 finished generating
Group 8 finished generating
Eval pairs written.


In [21]:
proportion_ci(0.80, 24)

(0.6399666701381658, 0.9600333298618343)

## Generate Training Pairs (run once)

In [ ]:
get_preference_pairs(175, "D:/Nanogpt-SFT/data/stories.jsonl", verbose=5, write_path = "D:/Nanogpt-SFT/data/preference_pairs.jsonl", return_pairs=False)

In [8]:
good_pairs, bad_pairs = [], []
with open("D:/Nanogpt-SFT/data/preference_pairs.jsonl") as f:
    for line in f:
        try:
            p = json.loads(line)
            good_pairs.append(p["good"])
            bad_pairs.append(p["bad"])
        except json.JSONDecodeError:
            pass
print(len(good_pairs), "pairs loaded")

477 pairs loaded


## Train Reward Model

In [11]:
reward_model = load_model(
    Transformer(vocab_size, 768, 6, 4, 8, rope=True, pre_ln=False, rms=True, swiglu=False, reward_model=True),
    "D:/Nanogpt-SFT/models/sft_model_500_147.npy",
    reward_model=True
)
loss_log = reward_model_train(reward_model, good_pairs, bad_pairs)
save_model(reward_model, "D:/Nanogpt-SFT/models/reward_model.npy")


Iter 0  loss: 0.7019  time: 1.724s
Iter 1  loss: 0.6593  time: 1.012s
Iter 2  loss: 0.5936  time: 0.871s
Iter 3  loss: 0.5950  time: 0.851s
Iter 4  loss: 0.4335  time: 0.832s
Iter 5  loss: 0.6023  time: 0.973s
Iter 6  loss: 0.8642  time: 0.917s
Iter 7  loss: 0.7507  time: 0.920s
Iter 8  loss: 0.8230  time: 0.935s
Iter 9  loss: 0.6483  time: 0.999s
Iter 10  loss: 0.7296  time: 0.960s
Iter 11  loss: 0.6946  time: 0.768s
Iter 12  loss: 0.6795  time: 0.875s
Iter 13  loss: 0.6960  time: 1.119s
Iter 14  loss: 0.7256  time: 1.272s
Iter 15  loss: 0.6993  time: 1.033s
Iter 16  loss: 0.6641  time: 0.995s
Iter 17  loss: 0.6392  time: 1.012s
Iter 18  loss: 0.7291  time: 0.994s
Iter 19  loss: 0.7047  time: 0.951s
Iter 20  loss: 0.5795  time: 0.991s
Iter 21  loss: 0.7282  time: 1.160s
Iter 22  loss: 0.5566  time: 0.943s
Iter 23  loss: 0.8527  time: 0.937s
Iter 24  loss: 0.7705  time: 1.003s
Iter 25  loss: 0.6035  time: 1.087s
Iter 26  loss: 0.4692  time: 1.036s
Iter 27  loss: 0.4680  time: 1.002s
It

## Eval — DONE Gate

In [2]:
import json
eval_good, eval_bad = [], []
with open("D:/Nanogpt-SFT/data/eval_preference_pairs.jsonl") as f:
    for line in f:
        try:
            p = json.loads(line)
            eval_good.append(p["good"])
            eval_bad.append(p["bad"])
        except json.JSONDecodeError:
            pass
print(len(eval_good), "eval pairs loaded.")


24 eval pairs loaded.


In [9]:
reward_model = load_model(
    Transformer(vocab_size, 768, 6, 4, 8, rope=True, pre_ln=False, rms=True, swiglu=False, reward_model=True),
    "D:/Nanogpt-SFT/models/reward_model.npy",
    reward_model=False
)

In [10]:
correct = 0
for g, b in zip(eval_good, eval_bad):
    out_g = reward_model.forward(g)
    out_b = reward_model.forward(b)
    r_g = float(out_g.data[0])
    r_b = float(out_b.data[0])
    print(r_g,r_b)
    collect(out_g); collect(out_b)
    if r_g > r_b:
        correct += 1

p_hat = correct / len(eval_good)
low, high = proportion_ci(p_hat, len(eval_good))
print(f"{correct}/{len(eval_good)} = {p_hat:.3f}")
print(f"95% CI: ({low:.3f}, {high:.3f})")
print(f"Excludes 0.5: {low > 0.5}")

3.840574264526367 2.5679216384887695
2.9744672775268555 2.929187774658203
3.3852875232696533 2.6431820392608643
2.8903050422668457 2.638162136077881
2.534675359725952 2.4440340995788574
3.2353100776672363 2.57479190826416
3.193756580352783 2.8080811500549316
3.7211337089538574 2.8050537109375
3.409158706665039 3.0287017822265625
3.028470516204834 2.6690900325775146
3.2168145179748535 3.029665470123291
4.307288646697998 4.2163405418396
3.346015453338623 2.8835251331329346
3.6510632038116455 2.7190041542053223
3.4590795040130615 3.6197009086608887
3.1047399044036865 3.091052770614624
3.185568332672119 2.5329322814941406
4.3357768058776855 3.229741334915161
2.842839479446411 2.681925058364868
2.9311728477478027 3.042402744293213
2.5959298610687256 2.6162993907928467
2.9023590087890625 2.779228448867798
4.068608283996582 2.7151761054992676
3.1715524196624756 2.5695629119873047
21/24 = 0.875
95% CI: (0.743, 1.000)
Excludes 0.5: True
